In [ ]:
# BA PROJECT 2026 - AI-DRIVEN TRIAGE SYSTEM FOR LOGISTICS DELAYS
# Theme: AI Solution for  logistic Industry


import pandas as pandas
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. LOAD DATA - Python and pandas to organise data 
df = pd.read_csv('synthetic_logistics_delay_triage.csv')
print(df['delay_label'].value_counts())

# 2. PREPARE - Don't use recommended_action as input 
X = df.drop(columns=['shipment_id', 'delay_label', 'recommended_action'])
y = df['delay_label']

numeric = ['distance_km','cargo_weight_kg','origin_congestion','dest_congestion',
           'carrier_reliability','scheduled_hours','delay_risk_score']
categorical = ['transport_mode','region','weather']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

# 3. SPLIT - 80% train, 20% test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                     random_state=42, stratify=y)

# 4. THREE MODELS FROM YOUR DOC 
models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree (Easy to explain)": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest (More reliable + important factors)": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
}

best_model = None
best_acc = 0

# 5. EVALUATION - Accuracy, Precision, Recall, Confusion Matrix + CV
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    cv = cross_val_score(pipe, X_train, y_train, cv=5).mean()
    
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f} | 5-Fold CV: {cv:.4f}")
    print(classification_report(y_test, pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    
    if acc > best_acc:
        best_acc = acc
        best_model = pipe

# 6. BUSINESS SOLUTION - Prioritise shipments
# This stops manual checking of 1,800 shipments
df['predicted'] = best_model.predict(X)
urgency = {'critical_delay':4, 'major_delay':3, 'minor_delay':2, 'on_time':1}
df['urgency_score'] = df['predicted'].map(urgency)

# Action as per AI Solution: monitor, reroute, hold, expedite
action_map = {'on_time':'monitor','minor_delay':'monitor','major_delay':'reroute','critical_delay':'expedite'}
df['suggested_action'] = df['predicted'].map(action_map)

# Correct organizing - descending order of importance
ranked = df.sort_values(by=['urgency_score','delay_risk_score'], ascending=False)

ranked.to_csv('prioritized_shipments_10marks.csv', index=False)
print("\nTop 10 urgent shipments for managers:")
print(ranked[['shipment_id','predicted','delay_risk_score','suggested_action']].head(10))

# 7. CHATBOT / NLP FEATURE 
def chatbot(q, data):
    q = q.lower()
    if "critical" in q or "vital" in q:l
        critical = data[data['predicted']=='critical_delay']
        return f"{len(critical)} critical shipments need expedite: {critical['shipment_id'].head(3).tolist()}"
    if "prevent" in q:
        return "Prevent by checking weather=storm + congestion>0.8 + carrier reliability low. Reroute early."
    return "Ask: 'which shipments have vital delays'"

print("\nChatbot:", chatbot("which shipments have vital delays", ranked))
